In [1]:
"""
===========================================================
SCRIPT 02
Buffer interno del Ground Truth
===========================================================

Objetivo
--------
Generar un Ground Truth optimizado para entrenamiento
eliminando los píxeles cercanos a los bordes de cada
polígono mediante un buffer interno de 5 metros.

Entrada
-------
GroundTruth_SAGAMI.gpkg

Salida
------
GroundTruth_SAGAMI_Buffer5m.gpkg

Autor:
Luis Miguel Gómez Meneses

===========================================================
"""

from pathlib import Path
import geopandas as gpd

# ==========================================================
# CONFIGURACIÓN
# ==========================================================

INPUT_FILE = Path("/content/GroundTruth_SAGAMI_.gpkg")

OUTPUT_FILE = Path("GroundTruth_SAGAMI_Buffer5m.gpkg")

BUFFER_DISTANCE = -5      # metros

# UTM Zona 19N (Arauca)
TARGET_CRS = "EPSG:32619"

# ==========================================================
# CARGAR GROUND TRUTH
# ==========================================================

print("="*60)
print("CARGANDO GROUND TRUTH")
print("="*60)

gdf = gpd.read_file(INPUT_FILE)

print(f"Polígonos originales : {len(gdf)}")
print(f"CRS original         : {gdf.crs}")

# ==========================================================
# REPROYECTAR A UTM
# ==========================================================

print("\nReproyectando a", TARGET_CRS)

gdf = gdf.to_crs(TARGET_CRS)

# ==========================================================
# CALCULAR ÁREA ORIGINAL
# ==========================================================

gdf["area_original_m2"] = gdf.area

# ==========================================================
# BUFFER INTERNO
# ==========================================================

print("\nAplicando buffer interno de 5 metros...")

gdf["geometry"] = gdf.buffer(BUFFER_DISTANCE)

# ==========================================================
# ELIMINAR GEOMETRÍAS VACÍAS
# ==========================================================

gdf = gdf[~gdf.geometry.is_empty]

gdf = gdf[gdf.geometry.notnull()]

# ==========================================================
# CORREGIR GEOMETRÍAS INVÁLIDAS
# ==========================================================

gdf["geometry"] = gdf.buffer(0)

gdf = gdf[gdf.is_valid]

# ==========================================================
# CALCULAR ÁREA FINAL
# ==========================================================

gdf["area_buffer_m2"] = gdf.area

gdf["area_perdida_m2"] = (
    gdf["area_original_m2"] -
    gdf["area_buffer_m2"]
)

gdf["porcentaje_perdido"] = (
    100 *
    gdf["area_perdida_m2"] /
    gdf["area_original_m2"]
)

# ==========================================================
# REPORTE
# ==========================================================

print("\n" + "="*60)
print("REPORTE")
print("="*60)

print(f"Polígonos finales : {len(gdf)}")

print()

print("Área total original (ha):",
      round(gdf["area_original_m2"].sum()/10000,2))

print("Área total final (ha):",
      round(gdf["area_buffer_m2"].sum()/10000,2))

print("Área eliminada (ha):",
      round(gdf["area_perdida_m2"].sum()/10000,2))

print()

print("Reducción media (%) :",
      round(gdf["porcentaje_perdido"].mean(),2))

# ==========================================================
# GUARDAR
# ==========================================================

gdf.to_file(
    OUTPUT_FILE,
    driver="GPKG"
)

print("\nArchivo generado:")

print(OUTPUT_FILE)

print("\nProceso finalizado correctamente.")

CARGANDO GROUND TRUTH
Polígonos originales : 71
CRS original         : EPSG:4326

Reproyectando a EPSG:32619

Aplicando buffer interno de 5 metros...

REPORTE
Polígonos finales : 67

Área total original (ha): 1770.86
Área total final (ha): 1656.93
Área eliminada (ha): 113.92

Reducción media (%) : 26.68

Archivo generado:
GroundTruth_SAGAMI_Buffer5m.gpkg

Proceso finalizado correctamente.
